# He init — Delving Deep into Rectifiers (2015)

_Papers · Foundations & Optimization_

**Initialize weights with variance 2/fan_in (not 1/fan_in) so signal variance stays stable through deep ReLU stacks — plus PReLU, a ReLU with a learned negative slope.**

---

This notebook is a practice scaffold for the **He init — Delving Deep into Rectifiers (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, math

# ---- He / Kaiming normal init from scratch (forward, fan_in, ReLU) ----
def my_he_normal_(w):
    """In-place He-normal init for a ReLU layer.  Var[w] = 2 / fan_in."""
    fan_in = w.shape[1]                 # nn.Linear weight is (out, in) -> fan_in = in
    std = math.sqrt(2.0 / fan_in)       # the He standard deviation
    with torch.no_grad():
        return w.normal_(0.0, std)      # zero-mean Gaussian, std = sqrt(2/fan_in)

# ---- THE ORACLE: my init must equal nn.init.kaiming_normal_ bit-for-bit ----
fan_out, fan_in = 512, 256
w_mine = torch.empty(fan_out, fan_in)
w_ref  = torch.empty(fan_out, fan_in)
torch.manual_seed(42); my_he_normal_(w_mine)                                   # same RNG state...
torch.manual_seed(42); nn.init.kaiming_normal_(w_ref, mode='fan_in',
                                               nonlinearity='relu')            # ...for both
print("target std sqrt(2/256) =", round(math.sqrt(2/256), 6))                  # 0.088388
print("allclose mine vs kaiming_normal_:", torch.allclose(w_mine, w_ref))      # True
print("empirical std mine:", round(w_mine.std().item(), 5))                    # ~0.0888

# ---- recompute the worked example ----
print("fan_in=256: Var=2/256 =", 2/256, " std =", round(math.sqrt(2/256), 6)) # 0.0078125, 0.088388
print("fan_in=512: Var=2/512 =", 2/512, " std =", round(math.sqrt(2/512), 6)) # 0.00390625, 0.0625

# ---- activation variance vs depth: He (2/n) vs Xavier-scale (1/n) on a ReLU stack ----
def run_stack(c, depth=20, width=512, n=4096, seed=0):
    torch.manual_seed(seed)
    x = torch.randn(n, width)                      # unit-variance input
    vars_ = [x.var().item()]
    for _ in range(depth):
        W = torch.randn(width, width) * math.sqrt(c / width)   # c=2 -> He, c=1 -> Xavier
        x = torch.relu(x @ W)
        vars_.append(x.var().item())
    return vars_

he  = run_stack(2.0)
xav = run_stack(1.0)
print("\nHe   (Var=2/n) variance @ layers 0,5,10,15,20:",
      [round(he[i], 4) for i in (0,5,10,15,20)])     # stays ~0.5-0.8
print("Xav  (Var=1/n) variance @ layers 0,5,10,15,20:",
      [round(xav[i], 6) for i in (0,5,10,15,20)])    # collapses toward 0

## Visualize the data & results

_Push a unit-variance input through a 20-layer ReLU stack (no training) and measure the activation variance at each layer: does He init (Var[w]=2/n) keep it stable while Xavier-scale (Var[w]=1/n) collapses it toward zero?_

In [ ]:
import torch, math
torch.manual_seed(0)

def run_stack(c, depth=20, width=512, n=4096):
    x = torch.randn(n, width)                    # unit-variance input
    vars_ = [x.var().item()]
    for _ in range(depth):
        W = torch.randn(width, width) * math.sqrt(c / width)   # c=2 He, c=1 Xavier
        x = torch.relu(x @ W)                    # ReLU deletes the negative half
        vars_.append(x.var().item())
    return vars_

he  = run_stack(2.0)
xav = run_stack(1.0)
print("He  layer 0,10,20:", [round(he[i],4)  for i in (0,10,20)])   # ~1.00, ~0.76, ~0.51
print("Xav layer 0,10,20:", [round(xav[i],7) for i in (0,10,20)])   # ~1.00, ~5e-4, ~5e-7

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute the He standard deviation for a $3\times3$ convolution with 128 input channels.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Fan-in $n=k^2 c = 3^2\cdot 128 = 9\cdot 128 = 1152$. — _Conv fan-in counts the whole receptive field, not just channels._
- Variance $=2/1152=0.001736\ldots$ — _He's $2/n_{\text{in}}$._
- Std $=\sqrt{2/1152}=0.04167\ldots$ — _Standard deviation is the square root of the variance._

**Answer:** $\sigma=\sqrt{2/1152}\approx 0.0417$. Note the Xavier value would be $\sqrt{1/1152}\approx 0.0295$, a factor $\sqrt2$ smaller — which would halve the activation variance at this layer.

</details>

**Problem 2.** A 20-layer ReLU net is initialized with Xavier's $\mathrm{Var}[w]=1/n$. By what factor is the activation variance scaled from input to output, and why is that a problem?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Per-layer multiplier under ReLU is $\tfrac12 n\cdot(1/n)=\tfrac12$. — _Eq. (8) with $\mathrm{Var}[w]=1/n$; the $\tfrac12$ is ReLU deleting half the signal._
- Over 20 layers the scaling is $(\tfrac12)^{20}$. — _Each layer multiplies the variance by $\tfrac12$._
- $(\tfrac12)^{20}=1/1{,}048{,}576\approx 9.5\times10^{-7}$. — _Roughly a millionth._

**Answer:** Activation variance shrinks by about $9.5\times10^{-7}$ — essentially to zero. With no signal reaching the later layers (and, symmetrically, no gradient reaching the early ones), the network cannot learn. He's $2/n$ makes the multiplier $1$, so variance is preserved. This is exactly what the CODEVIZ chart shows.

</details>

**Problem 3.** Ablation: in the deep-net training cell, swap He ($\mathrm{Var}[w]=2/n$) for Xavier-scale ($\mathrm{Var}[w]=1/n$) on a 15-layer ReLU classifier and compare final loss. What do you expect and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set every layer's init std to $\sqrt{1/n_{\text{in}}}$ instead of $\sqrt{2/n_{\text{in}}}$. — _Drops the factor of 2._
- Train both with the same SGD and binary cross-entropy loss for 200 steps. — _Only the init differs._
- Compare the final loss to the random-guess loss $\ln 2\approx 0.693$. — _A net that learns nothing stays near $\ln 2$._

**Answer:** He trains: in our run the loss drops to ~0.0005. Xavier-scale stalls near ~0.67 (≈ the random-guess $\ln 2$): activations vanish through 15 ReLU layers, so the gradient that reaches the early weights is negligible and they barely move. The only change is the factor of 2 in the initial variance, which confirms the paper's central claim. (Our small run, not the paper's number.)

</details>